## Quality Control - Oligo and Polymetastatic Patients Clinical Data ##

This notebook checks the consistency and quality of the clinical data for patients with oligometastatic or polymetastatic breast cancer.

## Inputs ##

- `mnt/data3/MCPP-Breast/Clinical-Data/20260406-mcpp-breast-clinical-data.csv`: clinical data of the whole cohort of the breast cancer dataset patients

## Import Libraries and Load Data ##

In [1]:
# Display precision only; analytical values and saved artifacts stay unchanged.
from pathlib import Path as _DisplayPath
import sys as _display_sys
_display_root = _DisplayPath.cwd()
while not (_display_root / "scripts").is_dir() and _display_root != _display_root.parent:
    _display_root = _display_root.parent
if str(_display_root) not in _display_sys.path:
    _display_sys.path.insert(0, str(_display_root))
from scripts.OligoPolyThesis.utils_ml_oligopoly import thesis_display, thesis_print
from scripts.OligoPolyThesis.utils_ml_oligopoly import validate_allowed_values

import os
import sys
import pandas as pd

path_directory = '/mnt/data3/MCPP-Breast/Clinical-Data/'
file_path = os.path.join(path_directory, '20260406-mcpp-breast-clinical-data.csv')

thesis_print(file_path)
thesis_print(os.path.exists(file_path))

df_clinical = pd.read_csv(file_path, sep=';')
thesis_print(df_clinical.shape)

/mnt/data3/MCPP-Breast/Clinical-Data/20260406-mcpp-breast-clinical-data.csv
True
(217, 1440)


In [2]:
# Standard Libraries
import re
import joblib

# Data Manipulation
import numpy as np


In [3]:
thesis_display(df_clinical.head(10))

,record_id,sample_id,FFPE_type,abc_date_of_birth,abc_smoker,abc_smoking_stopdate,abc_alcohol,abc_date_of_diagnosis,histo_result___1,abc_breast_side,...,oligo_to_pluri_date,involved_organs_pluri___1,involved_organs_pluri___2,involved_organs_pluri___3,involved_organs_pluri___4,involved_organs_pluri___5,involved_organs_pluri___6,involved_organs_pluri___7,last_fu,death_date
0,216,#196BC,Breast,26/09/1952,1.0,NaN,2.0,26/07/2002,1,1.0,...,NaN,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,06/12/2017
1,221,#197BC,Breast,17/02/1933,1.0,NaN,1.0,04/02/2002,1,1.0,...,NaN,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,30/03/2017
2,222,#198BC,Breast,01/01/1936,1.0,NaN,5.0,24/10/2002,1,1.0,...,NaN,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,16/02/2019
3,224,#202BC,Breast,06/12/1944,4.0,NaN,5.0,28/06/2004,1,3.0,...,NaN,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,17/05/2016
4,225,#203BC,Breast,17/06/1950,1.0,NaN,1.0,18/11/2005,1,1.0,...,NaN,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,25/08/2017
5,226,#205BC,Breast,08/05/1936,1.0,NaN,1.0,01/07/2006,1,2.0,...,NaN,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,01/05/2020
6,227,#207BC,Breast,15/08/1958,4.0,NaN,2.0,15/12/2005,1,1.0,...,NaN,0.0,0.0,0.0,0.0,0.0,0.0,0.0,14/02/2024,NaN
7,228,#208BC,Skin,16/01/1945,4.0,NaN,5.0,01/01/1998,1,2.0,...,NaN,0.0,0.0,0.0,0.0,0.0,0.0,0.0,13/10/2025,NaN
8,229,#209BC,Skin,19/05/1946,2.0,01/09/2007,5.0,02/08/2007,1,1.0,...,NaN,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,14/04/2014
9,230,#210BC,Breast,13/01/1955,1.0,NaN,1.0,15/11/2007,1,1.0,...,NaN,0.0,0.0,0.0,0.0,0.0,0.0,0.0,25/01/2024,NaN


## Data Preprocessing ##

### Convert date columns from string to datetime ###

In [4]:
# Convert date columns from string to datetime (format: dd/mm/yyyy)
date_columns = [
    'abc_date_of_birth', 
    'abc_date_of_diagnosis', 
    'abc_m1_diagnosis',
    'oligo_to_pluri_date',
    'death_date',
    'last_fu'
]
df_clinical[date_columns] = df_clinical[date_columns].apply(
    lambda x: pd.to_datetime(x, format='%d/%m/%Y', errors='coerce')
)

### Inconsistencies Check ###

#### Check Chronological Consistency Across All Clinical Dates ####

In [5]:
# ============================================================
# Check Chronological Consistency Across Clinical Dates
# Report all inconsistencies without removing patients
# ============================================================

# Make a fresh copy to avoid fragmentation warnings
df_clinical = df_clinical.copy()

# ------------------------------------------------------------
# 1. Define date columns
# ------------------------------------------------------------
birth_col = 'abc_date_of_birth'
primary_col = 'abc_date_of_diagnosis'
oligo_pluri_col = 'oligo_to_pluri_date'
death_col = 'death_date'
last_fu_col = 'last_fu'   

# ------------------------------------------------------------
# 2. Create final event date = death if available, otherwise last follow-up
#    and store which one was used
# ------------------------------------------------------------
df_clinical['final_event_date'] = df_clinical[death_col].combine_first(df_clinical[last_fu_col])

df_clinical['final_event_source'] = pd.NA
mask_death = df_clinical[death_col].notna()
mask_last = df_clinical[last_fu_col].notna()

df_clinical.loc[mask_death, 'final_event_source'] = death_col
df_clinical.loc[~mask_death & mask_last, 'final_event_source'] = last_fu_col

# ------------------------------------------------------------
# 3. Check main chronology by comparing consecutive NON-NULL dates in the expected sequence:
# birth <= primary <= m1 <= oligo_to_pluri <= final_event
# ------------------------------------------------------------
main_sequence = [
    birth_col,
    primary_col,
    'abc_m1_diagnosis',
    oligo_pluri_col,
    'final_event_date'
]

id_cols = [col for col in ['record_id', 'sample_id'] if col in df_clinical.columns]
all_inconsistencies = []

for idx, row in df_clinical.iterrows():
    available_dates = [(col, row[col]) for col in main_sequence if pd.notna(row[col])]
    
    for i in range(1, len(available_dates)):
        left_col, left_val = available_dates[i - 1]
        right_col, right_val = available_dates[i]
        
        if left_val > right_val:
            issue = {}
            for c in id_cols:
                issue[c] = row[c]
            issue['left_date_column'] = left_col
            issue['right_date_column'] = right_col
            issue['left_date_value'] = left_val
            issue['right_date_value'] = right_val
            issue['issue_description'] = f'{left_col} should be on or before {right_col}'
            issue['days_difference'] = (left_val - right_val).days
            issue['final_event_source'] = row['final_event_source']
            all_inconsistencies.append(issue)

# ------------------------------------------------------------
# 4. Build results dataframe
# ------------------------------------------------------------
if all_inconsistencies:
    date_order_issues = pd.DataFrame(all_inconsistencies)

    thesis_print(f"Total number of chronological inconsistencies found: {len(date_order_issues)}")
    if id_cols:
        thesis_print(f"Number of patients with at least one inconsistency: {date_order_issues[id_cols].drop_duplicates().shape[0]}")

    thesis_display(date_order_issues.head(50))

    summary_issues = (
        date_order_issues
        .groupby(['left_date_column', 'right_date_column', 'issue_description', 'final_event_source'])
        .size()
        .reset_index(name='count')
        .sort_values('count', ascending=False)
    )

    thesis_print("Summary of chronological inconsistencies:")
    thesis_display(summary_issues)

else:
    thesis_print("No chronological inconsistencies found.")

No chronological inconsistencies found.


#### Check if There Are Values Outside the Defined Options ####

In [6]:
invalid_values_df, summary_invalid, allowed_values_used = validate_allowed_values(df_clinical)

thesis_display(invalid_values_df.head(50))
thesis_display(summary_invalid)

No invalid entries found for the specified columns.


""


,invalid_column,normalized_invalid_value,count
